# QLoRA — Qwen2.5-1.5B para formulación LP/MILP

Este notebook entrena `Qwen/Qwen2.5-1.5B-Instruct` con QLoRA sobre el dataset
`OR-SFT v1.2-final`.

Modos:
- `pilot`: 160 ejemplos de 40 familias para train + 40 ejemplos de 10 familias no vistas para validación.
- `development`: 840 train + 210 validation, separados por familias.
- `final`: los 1050 ejemplos completos, una vez congelados los hiperparámetros.

El loss se calcula **solo sobre los tokens de la respuesta del assistant**.
Los checkpoints y el adapter final se guardan directamente en Google Drive.

In [ ]:
# ============================================================
# 1. INSTALAR DEPENDENCIAS
# ============================================================
!pip -q install -U "transformers>=4.48" "datasets>=3.0" "peft>=0.14" "accelerate>=1.2" "bitsandbytes>=0.45"

In [ ]:
# ============================================================
# 2. MONTAR GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# CAMBIA SOLO ESTA RUTA si tu carpeta en Drive tiene otro nombre.
PROJECT_DIR = Path("/content/drive/MyDrive/IA/qwen25_qlora_or")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_DIR)

## Cómo pasar el dataset

1. Descarga `or_sft_dataset_v1_2_final.zip` desde ChatGPT.
2. Súbelo a Google Drive, por ejemplo a:
   `Mi unidad/IA/or_sft_dataset_v1_2_final.zip`
3. Ajusta `ZIP_PATH` en la siguiente celda si usaste otra carpeta.
4. El notebook lo descomprime automáticamente en `/content/or_sft_dataset_v1_2_final`.

El ZIP original queda persistente en Drive. Los datos se copian al almacenamiento local
de Colab para leerlos más rápido.

In [ ]:
# ============================================================
# 3. LOCALIZAR Y EXTRAER EL DATASET DESDE DRIVE
# ============================================================
import zipfile
import shutil

ZIP_PATH = Path("/content/drive/MyDrive/IA/or_sft_dataset_v1_2_final.zip")
LOCAL_DATA_DIR = Path("/content/or_sft_dataset_v1_2_final")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro el dataset en:\n{ZIP_PATH}\n\n"
        "Descarga el ZIP desde ChatGPT, súbelo a Drive y corrige ZIP_PATH."
    )

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(LOCAL_DATA_DIR)

print("Dataset extraído en:", LOCAL_DATA_DIR)
print("\nArchivos:")
for p in sorted(LOCAL_DATA_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
# ============================================================
# 4. CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================

# Cambiar únicamente cuando corresponda:
RUN_MODE = "pilot"
# Opciones: "pilot", "development", "final"

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Longitud máxima total: system + user + assistant.
# El notebook medirá longitudes antes de entrenar.
MAX_LENGTH = 1024

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Hiperparámetros iniciales.
# No ajustar mirando el benchmark oficial de 30 casos.
LEARNING_RATE = 2e-4
GRAD_ACCUM = 8

if RUN_MODE == "pilot":
    NUM_EPOCHS = 1
    SAVE_STEPS = 10
elif RUN_MODE == "development":
    NUM_EPOCHS = 2
    SAVE_STEPS = 25
elif RUN_MODE == "final":
    NUM_EPOCHS = 2   # usar aquí el valor congelado tras development
    SAVE_STEPS = 25
else:
    raise ValueError("RUN_MODE debe ser pilot, development o final")

RUN_DIR = PROJECT_DIR / RUN_MODE
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FINAL_ADAPTER_DIR = RUN_DIR / "final_adapter"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print("Modo:", RUN_MODE)
print("Checkpoints:", CHECKPOINT_DIR)
print("Adapter final:", FINAL_ADAPTER_DIR)

In [ ]:
# ============================================================
# 5. CARGAR JSONL Y CONSTRUIR SPLIT
# ============================================================
import json
from collections import Counter, defaultdict
from datasets import Dataset

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train840 = read_jsonl(
    LOCAL_DATA_DIR / "or_sft_train_840_family_split_v1_2.jsonl"
)
val210 = read_jsonl(
    LOCAL_DATA_DIR / "or_sft_val_210_family_split_v1_2.jsonl"
)
all1050 = read_jsonl(
    LOCAL_DATA_DIR / "or_sft_1050_messages_v1_2.jsonl"
)

def first_n_per_family(rows, n):
    by_family = defaultdict(list)
    for row in rows:
        by_family[row["family_id"]].append(row)

    out = []
    for family_id in sorted(by_family):
        out.extend(by_family[family_id][:n])
    return out

if RUN_MODE == "pilot":
    # 4 muestras de cada una de las 40 familias de train = 160
    # 4 muestras de cada una de las 10 familias holdout = 40
    train_rows = first_n_per_family(train840, 4)
    eval_rows = first_n_per_family(val210, 4)

elif RUN_MODE == "development":
    train_rows = train840
    eval_rows = val210

else:  # final
    train_rows = all1050
    eval_rows = []

print("Train:", len(train_rows))
print("Eval:", len(eval_rows))

print("\nFamilias train:", len(set(x["family_id"] for x in train_rows)))
print("Familias eval :", len(set(x["family_id"] for x in eval_rows)))

overlap = (
    set(x["family_id"] for x in train_rows)
    & set(x["family_id"] for x in eval_rows)
)
print("Familias compartidas train/eval:", overlap)

if RUN_MODE != "final":
    assert len(overlap) == 0, "Hay fuga de familias entre train y eval."

In [ ]:
# ============================================================
# 6. CARGAR TOKENIZER Y MEDIR LONGITUDES
# ============================================================
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def chat_to_ids(messages, add_generation_prompt=False):
    """
    Conversión robusta:
    1) renderiza el chat como texto;
    2) tokeniza explícitamente;
    3) siempre devuelve list[int], nunca tensor/lista anidada.
    """
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
    )

    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

    # Defensa adicional por compatibilidad de versiones.
    if len(ids) > 0 and isinstance(ids[0], list):
        if len(ids) != 1:
            raise RuntimeError("Tokenización devolvió un batch inesperado.")
        ids = ids[0]

    return list(ids)


def full_token_length(row):
    return len(
        chat_to_ids(
            row["messages"],
            add_generation_prompt=False,
        )
    )


train_lengths = [full_token_length(x) for x in train_rows]
eval_lengths = [full_token_length(x) for x in eval_rows] if eval_rows else []


def describe_lengths(name, xs):
    if not xs:
        return

    arr = np.array(xs)

    print(
        f"{name}: n={len(xs)} | "
        f"media={arr.mean():.1f} | "
        f"p95={np.percentile(arr,95):.0f} | "
        f"p99={np.percentile(arr,99):.0f} | "
        f"max={arr.max()} | "
        f">MAX_LENGTH={(arr > MAX_LENGTH).sum()}"
    )


describe_lengths("Train", train_lengths)
describe_lengths("Eval ", eval_lengths)

print("\nTipo de un ejemplo tokenizado:", type(chat_to_ids(train_rows[0]["messages"])))
print("Primeros tokens:", chat_to_ids(train_rows[0]["messages"])[:10])

if any(x > MAX_LENGTH for x in train_lengths):
    print(
        "\nADVERTENCIA: hay ejemplos truncables. "
        "Antes de entrenar, considera subir MAX_LENGTH a 1280 o 1536 "
        "si la VRAM lo permite."
    )

In [ ]:
# ============================================================
# 7. TOKENIZACIÓN CON LOSS SOLO EN EL ASSISTANT
# ============================================================

def tokenize_record(row):
    messages = row["messages"]

    if len(messages) != 3:
        raise ValueError(
            f"{row['id']}: se esperaban exactamente system, user y assistant."
        )

    if [m["role"] for m in messages] != ["system", "user", "assistant"]:
        raise ValueError(
            f"{row['id']}: orden de roles inesperado."
        )

    # --------------------------------------------------------
    # 1) Prompt de entrada = system + user + cabecera assistant
    # --------------------------------------------------------
    prefix_text = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=False,
        add_generation_prompt=True,
    )

    # --------------------------------------------------------
    # 2) Conversación completa = prompt + target assistant
    # --------------------------------------------------------
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prefix_ids = tokenizer(
        prefix_text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

    # Compatibilidad defensiva frente a listas anidadas.
    if len(prefix_ids) > 0 and isinstance(prefix_ids[0], list):
        if len(prefix_ids) != 1:
            raise RuntimeError(f"{row['id']}: prefix batch inesperado.")
        prefix_ids = prefix_ids[0]

    if len(full_ids) > 0 and isinstance(full_ids[0], list):
        if len(full_ids) != 1:
            raise RuntimeError(f"{row['id']}: full batch inesperado.")
        full_ids = full_ids[0]

    prefix_ids = list(prefix_ids)
    full_ids = list(full_ids)

    # --------------------------------------------------------
    # 3) Determinar exactamente dónde empieza la respuesta
    # --------------------------------------------------------
    common = 0
    limit = min(len(prefix_ids), len(full_ids))

    while common < limit and prefix_ids[common] == full_ids[common]:
        common += 1

    # Qwen debería compartir prácticamente todo el prefix.
    if common < len(prefix_ids) - 2:
        raise RuntimeError(
            f"{row['id']}: el chat template no conserva el prefix. "
            f"Coinciden {common}/{len(prefix_ids)} tokens."
        )

    assistant_start = common

    # --------------------------------------------------------
    # 4) Truncar después de construir correctamente la máscara
    # --------------------------------------------------------
    input_ids = full_ids[:MAX_LENGTH]
    attention_mask = [1] * len(input_ids)

    labels = input_ids.copy()

    mask_until = min(assistant_start, len(labels))
    labels[:mask_until] = [-100] * mask_until

    supervised_tokens = sum(v != -100 for v in labels)
    masked_tokens = sum(v == -100 for v in labels)

    # Si queda demasiado poco target, el ejemplo no sirve.
    if supervised_tokens < 8:
        raise RuntimeError(
            f"{row['id']}: solo quedan {supervised_tokens} tokens "
            f"del assistant después del truncamiento. "
            f"prefix={assistant_start}, full={len(full_ids)}, "
            f"MAX_LENGTH={MAX_LENGTH}"
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "supervised_tokens": supervised_tokens,
        "masked_tokens": masked_tokens,
        "total_tokens": len(input_ids),
        "original_total_tokens": len(full_ids),
        "assistant_start": assistant_start,
        "was_truncated": len(full_ids) > MAX_LENGTH,
    }


train_tokenized = [tokenize_record(x) for x in train_rows]
eval_tokenized = [tokenize_record(x) for x in eval_rows] if eval_rows else []

print("Tokenización completada.")

train_sup = np.array([x["supervised_tokens"] for x in train_tokenized])
train_mask = np.array([x["masked_tokens"] for x in train_tokenized])
train_total = np.array([x["total_tokens"] for x in train_tokenized])

print(
    "Tokens supervisados train — "
    f"media={train_sup.mean():.1f}, "
    f"min={train_sup.min()}, "
    f"p95={np.percentile(train_sup,95):.0f}, "
    f"max={train_sup.max()}"
)

print(
    "Tokens enmascarados train — "
    f"media={train_mask.mean():.1f}, "
    f"min={train_mask.min()}, "
    f"max={train_mask.max()}"
)

print(
    "Ejemplos truncados train:",
    sum(x["was_truncated"] for x in train_tokenized),
    "/",
    len(train_tokenized),
)

if eval_tokenized:
    print(
        "Tokens supervisados eval — media:",
        round(np.mean([x["supervised_tokens"] for x in eval_tokenized]), 1)
    )
    print(
        "Ejemplos truncados eval:",
        sum(x["was_truncated"] for x in eval_tokenized),
        "/",
        len(eval_tokenized),
    )

# ------------------------------------------------------------
# 5) Verificación crítica de la máscara
# ------------------------------------------------------------
for row, x in zip(train_rows, train_tokenized):
    assert x["masked_tokens"] > 0, (
        f"{row['id']}: no hay tokens enmascarados."
    )
    assert x["supervised_tokens"] > 0, (
        f"{row['id']}: no hay tokens supervisados."
    )
    assert len(x["input_ids"]) == len(x["labels"]), (
        f"{row['id']}: input_ids y labels tienen distinta longitud."
    )

print("Máscara assistant-only: OK")

# Mostrar un diagnóstico del primer ejemplo.
x0 = train_tokenized[0]
r0 = train_rows[0]

print("\nDiagnóstico primer ejemplo:")
print("ID:", r0["id"])
print("Total original:", x0["original_total_tokens"])
print("Total usado:", x0["total_tokens"])
print("Inicio assistant:", x0["assistant_start"])
print("Tokens masked:", x0["masked_tokens"])
print("Tokens supervised:", x0["supervised_tokens"])

In [ ]:
# ============================================================
# 8. CREAR DATASETS HF
# ============================================================
from datasets import Dataset

train_ds = Dataset.from_list(train_tokenized)
eval_ds = Dataset.from_list(eval_tokenized) if eval_tokenized else None

print(train_ds)
if eval_ds is not None:
    print(eval_ds)

In [ ]:
# ============================================================
# 9. DATA COLLATOR PARA PADDEAR LABELS CON -100
# ============================================================
import torch

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            n_pad = max_len - len(f["input_ids"])

            input_ids.append(
                f["input_ids"]
                + [self.tokenizer.pad_token_id] * n_pad
            )
            attention_mask.append(
                f["attention_mask"] + [0] * n_pad
            )
            labels.append(
                f["labels"] + [-100] * n_pad
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CausalLMCollator(tokenizer)

In [ ]:
# ============================================================
# 10. CARGAR QWEN EN 4-BIT + PREPARAR QLoRA
# ============================================================
import gc
import torch

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Reduce memoria durante backprop.
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print(
    "VRAM asignada:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

In [ ]:
# ============================================================
# 11. TRAINING ARGUMENTS
# ============================================================
import inspect
from transformers import TrainingArguments

kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    logging_steps=5,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    load_best_model_at_end=False,

    report_to="none",
    remove_unused_columns=False,

    dataloader_pin_memory=True,
)

# Compatibilidad entre versiones de Transformers:
sig = inspect.signature(TrainingArguments.__init__)

if eval_ds is not None:
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs["evaluation_strategy"] = "steps"

    kwargs["eval_steps"] = SAVE_STEPS

training_args = TrainingArguments(**kwargs)

print(training_args)

In [ ]:
# ============================================================
# 12. CREAR TRAINER Y DETECTAR CHECKPOINT EN DRIVE
# ============================================================
from transformers import Trainer
from transformers.trainer_utils import get_last_checkpoint

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

last_checkpoint = None

if CHECKPOINT_DIR.exists():
    try:
        last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
    except Exception:
        last_checkpoint = None

print("Último checkpoint:", last_checkpoint)

In [ ]:
# ============================================================
# 13. ENTRENAR / REANUDAR
# ============================================================

# Si Colab se cayó y existe un checkpoint válido en Drive,
# esta celda continúa desde allí.
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

print(train_result)

In [ ]:
# ============================================================
# 14. GUARDAR ADAPTER FINAL Y TOKENIZER EN DRIVE
# ============================================================

trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

metrics = train_result.metrics

with open(RUN_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Adapter guardado en:", FINAL_ADAPTER_DIR)
print("Métricas guardadas en:", RUN_DIR / "train_metrics.json")

In [ ]:
# ============================================================
# 15. EVALUACIÓN DE LOSS EN FAMILIAS NO VISTAS
# ============================================================

if eval_ds is not None:
    eval_metrics = trainer.evaluate()

    with open(RUN_DIR / "eval_metrics.json", "w", encoding="utf-8") as f:
        json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

    print(eval_metrics)
else:
    print("RUN_MODE='final': no hay conjunto de validación.")

In [ ]:
# ============================================================
# 16. GENERAR 10 CASOS DE VALIDACIÓN SIN MOSTRAR EL GOLD
# ============================================================
import time

@torch.inference_mode()
def generate_formula(problem, max_new_tokens=600):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = out[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

if eval_rows:
    sample_rows = eval_rows[:10]

    generations = []

    for row in sample_rows:
        user_problem = row["messages"][1]["content"]

        candidate = generate_formula(user_problem)

        generations.append({
            "id": row["id"],
            "family": row["family"],
            "candidate": candidate,
        })

        print("\n" + "="*100)
        print(row["id"], "|", row["family"])
        print("="*100)
        print(candidate)

    with open(
        RUN_DIR / "validation_generations_no_gold.jsonl",
        "w",
        encoding="utf-8",
    ) as f:
        for row in generations:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(
        "\nGeneraciones guardadas en:",
        RUN_DIR / "validation_generations_no_gold.jsonl"
    )
else:
    print("Sin validación en RUN_MODE='final'.")

In [ ]:
# ============================================================
# 17. CHEQUEOS RÁPIDOS DE FORMATO EN LAS GENERACIONES
# ============================================================

if eval_rows:
    required_sections = [
        "Conjuntos",
        "Parámetros",
        "Variables",
        "Función objetivo",
        "Restricciones",
    ]

    def format_score(text):
        return sum(
            section.lower() in text.lower()
            for section in required_sections
        )

    for g in generations:
        g["format_score_5"] = format_score(g["candidate"])
        g["tokens_est"] = len(
            tokenizer(
                g["candidate"],
                add_special_tokens=False,
            )["input_ids"]
        )

    scores = [g["format_score_5"] for g in generations]

    print("Format score promedio /5:", round(sum(scores)/len(scores), 2))
    print("Casos con 5/5 secciones:", sum(s == 5 for s in scores), "/", len(scores))

## Después del piloto

No cambies todavía el benchmark oficial de 30 problemas ni el juez FOM-5.

Primero revisa:
1. que `train_loss` disminuya;
2. que `eval_loss` sea finito y razonable;
3. que las generaciones de las 10 familias no vistas sean compactas y paramétricas;
4. que no aparezcan ciclos de repetición ni truncamientos frecuentes.

Si el piloto muestra una señal positiva, cambia `RUN_MODE = "development"` y reinicia el runtime.
Después de congelar hiperparámetros con 840/210, usa `RUN_MODE = "final"` para entrenar
el adapter definitivo con los 1050 ejemplos y recién entonces evalúalo en los 30 casos oficiales.